# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: fetch latest code</h2>
            <span style="color:#f71;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull and merge your changes as needed</a>. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/><br/>
            After you've pulled the code, from the llm_engineering directory, in a Cursor Terminal, run:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [2]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
open_router_api_key = os.getenv('OPEN_ROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if open_router_api_key:
    print(f"OpenRouter API Key exists and begins {open_router_api_key[:4]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Grok API Key not set (and this is optional)
Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-o


In [4]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
open_router_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
open_router = OpenAI(api_key=open_router_api_key, base_url=open_router_url)

In [5]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"
OPEN_ROUTER_MODEL = "nvidia/nemotron-3-nano-30b-a3b:free"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"

## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [17]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26100',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i5-11300H @ 3.10GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (x86_64-posix-seh-rev0, Built by MinGW-W64 project) 8.1.0',
   'g++': 'g++.EXE (x86_64-posix-seh-rev0, Built by MinGW-W64 project) 8.1.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': 'cmake version 3.30.4', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [12]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = open_router.chat.completions.create(model=OPEN_ROUTER_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

## 1️⃣ Do you need to install a C++ compiler?

**No – you already have one.**  
The *toolchain* section of your report shows that both **gcc** and **g++** (MinGW‑W64) are present:

```
'gcc':  'gcc.EXE (x86_64-posix-seh-rev0, Built by MinGW-W64 project) 8.1.0'
'g++':  'g++.EXE (x86_64-posix-seh-rev0, Built by MinGW-W64 project) 8.1.0'
```

That means the compiler is already on your **PATH** and can be invoked directly from a command prompt (or from Python via `subprocess`).  
No extra installation steps are required.

> **If you ever lost the toolchain** (e.g., after a system refresh) you can install it in a single step with `winget`:
> ```powershell
> winget install --id=GNU.GCC -e --scope=user
> ```
> (MinGW‑W64 is the package that provides `gcc.exe` and `g++.exe`.)

---

## 2️⃣ Compile & run a single source file from Python

Below is a **complete, ready‑to‑copy** snippet that:

1. Compiles `main.cpp` **with the highest optimisation level** (`-O3`) and **native‑CPU flags** (`-march=native`) so the generated binary runs as fast as possible.
2. Executes the resulting executable.
3. Returns **stdout** (or raises an exception if something went wrong).

```python
import subprocess
from pathlib import Path

# ----------------------------------------------------------------------
# 1️⃣  Build command (fastest possible single‑file compilation)
# ----------------------------------------------------------------------
#   -O3          : maximal optimisation
#   -march=native: schedule instructions for the host CPU (i5‑11300H)
#   -static      : link the C++ runtime statically (makes the exe self‑contained)
#   -pipe        : avoid unnecessary file‑IO for intermediate stages
#   -std=c++20   : use the latest language standard (adjust if you need C++17 etc.)
#   main.cpp    : source file
#   -o main.exe : name of the produced executable
compile_cmd = [
    "g++",
    "-O3",
    "-march=native",
    "-static",
    "-pipe",
    "-std=c++20",
    "-o",
    "main.exe",        # output file (Windows executable suffix .exe)
    str(Path("main.cpp")),   # source file (absolute or relative path)
]

# ----------------------------------------------------------------------
# 2️⃣  Run command
# ----------------------------------------------------------------------
run_cmd = ["main.exe"]      # just execute the compiled binary

# ----------------------------------------------------------------------
# 3️⃣  Helper that does the whole thing and returns stdout
# ----------------------------------------------------------------------
def compile_and_run(source_path: str) -> str:
    """
    Compiles the given C++ source file with maximal speed options
    and runs the resulting executable, returning its standard output.
    If anything fails a CalledProcessError is raised.
    """
    # ---- compile --------------------------------------------------------
    compile_res = subprocess.run(
        compile_cmd,
        check=True,
        text=True,
        capture_output=True,
    )
    # At this point `main.exe` exists

    # ---- run ------------------------------------------------------------
    run_res = subprocess.run(
        run_cmd,
        check=True,
        text=True,
        capture_output=True,
    )
    return run_res.stdout.strip()   # strip trailing newline / whitespace


# ----------------------------------------------------------------------
# Example usage ---------------------------------------------------------
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Assume `main.cpp` lives in the current working directory
    result = compile_and_run("main.cpp")
    print("=== program output ===")
    print(result)
```

### How it works

| Step | What happens | Why this is the *fastest* setting |
|------|--------------|-----------------------------------|
| **Compilation** | `g++ -O3 -march=native -static -pipe -std=c++20 -o main.exe main.cpp` | `-O3` = highest optimisation level; `-march=native` tells the compiler to emit instructions tuned for your exact CPU (i5‑11300H); `-static` bundles the C++ runtime so you don’t need any extra DLLs; `-pipe` removes temporary file I/O; `-std=c++20` selects the newest, best‑optimised language features. |
| **Execution** | `subprocess.run(["main.exe"], …)` | Calls the native binary directly; no extra shell, no extra path gymnastics — just one process that runs the compiled program. |
| **Return value** | `run_res.stdout` (stripped) | Gives you the exact text printed by the program, ready for further processing. If compilation or execution fails, `subprocess.run(..., check=True)` raises a `CalledProcessError`, which you can catch if needed. |

---

## 3️⃣ Quick **manual** sanity‑check (just to be sure everything works)

Open a **Command Prompt** (or PowerShell) and run the following two commands (replace the path to `main.cpp` if it isn’t in the current folder):

```cmd
rem --- 1️⃣ compile -------------------------------------------------
g++ -O3 -march=native -static -pipe -std=c++20 -o main.exe main.cpp

rem --- 2️⃣ run -----------------------------------------------------
main.exe
```

If the source program prints something, you should see that output right away.  
If you see an error like *“g++ is not recognized”* then the compiler is not on the PATH – in that case install it with the `winget` command shown earlier.

---

### TL;DR

*You already have a working C++ toolchain (g++). No extra installation steps are required.*  
Use the Python snippet above; the compile command is:

```python
compile_cmd = ["g++", "-O3", "-march=native", "-static", "-pipe", "-std=c++20", "-o", "main.exe", "main.cpp"]
```

and the run command is simply:

```python
run_cmd = ["main.exe"]
```

Running those via `subprocess.run` with `check=True` will give you the compiled binary’s stdout in the fastest possible way on your Windows 11 machine. Happy coding! 🚀

## If you need to install something

If you would like to, please follow GPTs instructions! Then rerun the analysis afterwards (you might need to Restart the notebook) to confirm you're set.

You should now be equipped with the command to compile the code, and the command to run it!

Enter that in the cell below:

In [25]:
# compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
# run_command = ["./main"]

compile_command = ["g++", "-O3", "-march=native", "-static", "-pipe", "-std=c++17", "-o", "main.exe", "main.cpp"]
run_command = ["main.exe"]


## And now, on with the main task

In [6]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [7]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [8]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [9]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [10]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [11]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [12]:
run_python(pi)

Result: 3.141592656089
Execution Time: 41.168631 seconds


In [18]:
port(open_router, OPEN_ROUTER_MODEL, pi)

# Compiling C++ and executing

This next cell contains the command to compile a C++ file based on the instructions from GPT.

Again, it's not crucial to do this step if you don't wish to!

OR alternatively: student Sandeep K.G. points out that you can run Python and C++ code online to test it out that way. Thank you Sandeep!  
> Not an exact comparison but you can still get the idea of performance difference.  
> For example here: https://www.programiz.com/cpp-programming/online-compiler/

In [26]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [27]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.417481 seconds

Result: 3.141592656089
Execution Time: 0.410985 seconds

Result: 3.141592656089
Execution Time: 0.409267 seconds



In [28]:
19.178207/0.082168

# For me the time take to run is
41.168631/ 0.410985 


100.17064126427971

## OK let's try the other contenders!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [29]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")


In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: 184X speedup
3rd place: GPT-5: 233X speedup
2nd place: Grok 4: 1060X speedup
1st place: Gemini 2.5 Pro: 1440X speedup

